# NERA Risk Intelligence — 03. Tabular XGBoost Model & SHAP Attribution
### Temporal Validation Split (Train: 2007–2013 | Val: 2014 | Test: 2015–2016)

Trains the primary environmental risk model using **16 real + derived features**:
- **Meteorological**: `rainfall_24h`, `rainfall_72h`, `rainfall_7d`, `rainfall_anomaly`
- **Geomorphic**: `elevation`, `slope`, `terrain_ruggedness`
- **Hydrological / Land**: `distance_to_river`, `land_cover`, `water_occurrence`
- **Geological / Infrastructure**: `historical_landslide_count`, `highway_class_encoded`, `bridge`, `road_length_m`
- **Seasonal**: `month`, `monsoon_indicator`

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import average_precision_score, roc_auc_score, classification_report, precision_recall_curve
import matplotlib.pyplot as plt
import joblib
import json
from pathlib import Path

# Load Preprocessed Dataset
DATA_DIR = Path("../python-service/data/features")
df = pd.read_parquet(DATA_DIR / "ml_dataset.parquet")
print(f"Loaded dataset: {df.shape[0]} rows x {df.shape[1]} columns")

In [ ]:
FEATURES = [
    "rainfall_24h", "rainfall_72h", "rainfall_7d", "rainfall_anomaly",
    "elevation", "slope", "terrain_ruggedness",
    "distance_to_river", "land_cover", "water_occurrence",
    "historical_landslide_count", "highway_class_encoded", "bridge",
    "road_length_m", "month", "monsoon_indicator"
]
TARGET = "disruption_next_24h"

# Temporal Split (Strict Past -> Future)
train_mask = df["split"] == "train"
val_mask = df["split"] == "val"
test_mask = df["split"] == "test"

X_train, y_train = df.loc[train_mask, FEATURES], df.loc[train_mask, TARGET]
X_val, y_val = df.loc[val_mask, FEATURES], df.loc[val_mask, TARGET]
X_test, y_test = df.loc[test_mask, FEATURES], df.loc[test_mask, TARGET]

print(f"Train: {len(X_train)} (Pos: {y_train.sum()}) | Val: {len(X_val)} (Pos: {y_val.sum()}) | Test: {len(X_test)} (Pos: {y_test.sum()})")

In [ ]:
# Class Imbalance Ratio
pos_count = max(1, y_train.sum())
scale_pos_weight = (len(y_train) - pos_count) / pos_count

# Train XGBoost Classifier
model = xgb.XGBClassifier(
    n_estimators=400,
    learning_rate=0.03,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    eval_metric="aucpr",
    random_state=42
)
model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=50)

# Isotonic Probability Calibration
calibrator = CalibratedClassifierCV(estimator=model, method="isotonic", cv="prefit")
calibrator.fit(X_val, y_val)

# Evaluate on Out-of-Time Test Set
y_prob = calibrator.predict_proba(X_test)[:, 1]
pr_auc = average_precision_score(y_test, y_prob)
roc_auc = roc_auc_score(y_test, y_prob)

print(f"\n=== Out-of-Time Test Evaluation (2015-2016) ===")
print(f"PR-AUC (Primary Metric): {pr_auc:.4f}")
print(f"ROC-AUC:                {roc_auc:.4f}")

## Export Model Artifact for FastAPI Serving

In [ ]:
MODEL_OUT = Path("../python-service/models")
MODEL_OUT.mkdir(parents=True, exist_ok=True)
joblib.dump(model, MODEL_OUT / "disruption_xgb.joblib")
joblib.dump(calibrator, MODEL_OUT / "calibrator.joblib")
print(f"Exported model to {MODEL_OUT}/disruption_xgb.joblib")